# Avaliação de Qualidade (Gen AI Evaluation) - Gemini Tuned para FHIR

Este notebook permite executar, gerenciar e visualizar de forma rica os resultados de qualidade das conversões de relatórios clínicos em português para recursos FHIR encapsulados em Bundles de transação.

In [36]:
# Instalação de dependências
!pip install -q "google-cloud-aiplatform[evaluation]>=1.111.0" pandas


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [37]:
import os
import json
import pandas as pd
from vertexai import Client, types

# Ajusta diretórios
WORKSPACE_DIR = os.getcwd()
DATASET_PATH = os.path.join(WORKSPACE_DIR, "evaluation_dataset_flat.jsonl")

print(f"Diretório do Workspace: {WORKSPACE_DIR}")
print(f"Caminho do Dataset: {DATASET_PATH}")

Diretório do Workspace: /Users/amandafurtado/dev/fine-tuning-hcls
Caminho do Dataset: /Users/amandafurtado/dev/fine-tuning-hcls/evaluation_dataset_flat.jsonl


In [ ]:
# Inicializa o cliente Google Gen AI

PROJECT_ID = ""  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))
LOCATION= "us-central1"  # @param {type: "string", placeholder: "us-central1", isTemplate: true}
LOCATION = os.environ.get("GOOGLE_CLOUD_REGION", LOCATION)
client = Client(project=PROJECT_ID, location=LOCATION)

In [39]:
# Carrega o dataset de avaliação gerado
if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(f"Dataset não encontrado em {DATASET_PATH}. Por favor, execute o script generate_eval_datasets.py primeiro.")

df = pd.read_json(DATASET_PATH, lines=True)
print(f"Dataset carregado com sucesso! Quantidade de registros: {len(df)}")

# Limitando a 3 exemplos
df = df[:3]
print(f"Dataset limitado a 3 exemplos! Quantidade de registros: {len(df)}")

# Caso use métricas de computação
df['reference'] = df['response']

Dataset carregado com sucesso! Quantidade de registros: 15
Dataset limitado a 3 exemplos! Quantidade de registros: 3


In [40]:
def validate_fhir_bundle_syntax(instance: dict) -> dict:
    """
    Métrica customizada programática para validar a sintaxe JSON
    e certificar se a resposta gerada é um Bundle FHIR válido.
    Retorna 1.0 se passar na estrutura básica de Bundle, caso contrário 0.0.
    """
    import json
    response_val = instance.get("response", "")
    
    # Extract text if response is in Gemini parts format
    if isinstance(response_val, dict):
        parts = response_val.get("parts", [])
        if parts and isinstance(parts, list):
            text_val = parts[0].get("text", "")
            if isinstance(text_val, str):
                response_val = text_val

    if isinstance(response_val, dict):
        data = response_val
    elif isinstance(response_val, str):
        response_text = response_val.strip()
        if not response_text:
            return {"score": 0.0, "explanation": "Resposta vazia."}
        try:
            data = json.loads(response_text)
        except json.JSONDecodeError as e:
            return {"score": 0.0, "explanation": f"Falha no parse de JSON: {str(e)}"}
    else:
        return {"score": 0.0, "explanation": f"Tipo de resposta inválido: {type(response_val)}"}
        
    # Valida se é do tipo Bundle
    if data.get("resourceType") != "Bundle":
        return {"score": 0.0, "explanation": "JSON válido, mas o resourceType não é 'Bundle'."}
        
    if "entry" not in data or not isinstance(data["entry"], list):
        return {"score": 0.5, "explanation": "Resource é Bundle, mas não contém a lista 'entry' de recursos."}
        
    return {"score": 1.0, "explanation": "Bundle FHIR estruturalmente válido (JSON parseado com sucesso)."}

In [41]:
# Rúbrica estática customizada
fhir_clinical_rubric = types.LLMMetric(
    name="fhir_clinical_completeness",
    prompt_template=types.MetricPromptBuilder(
        instruction=(
            "Avalie a completude clínica do Bundle FHIR com base no prontuário fornecido. "
            "Você DEVE retornar a resposta em formato JSON válido, contendo obrigatoriamente duas chaves: "
            "'score' (um número inteiro de 1 a 5 que representa a nota da avaliação) e "
            "'explanation' (uma string contendo a justificativa detalhada em texto estruturado). "
            "ATENÇÃO: NÃO utilize aspas duplas (\") no corpo do texto da justificativa ('explanation') "
            "para não corromper a sintaxe do JSON. Se necessário, utilize aspas simples (')."
        ),
        criteria={
            "Fidelidade de Dados": "Todas as observações de exames físicos e sintomas citados no prontuário foram convertidos?",
            "Uso de Padrões": "Todos os medicamentos e diagnósticos utilizam os sistemas corretos (RxNorm e SNOMED-CT)?",
            "Formatação JSON": "O código FHIR final gerado está limpo e bem-formado?"
        },
        rating_scores={
            "5": "Excelente: O Bundle está completo, todos os recursos clínicos foram modelados e codificados corretamente.",
            "4": "Bom: Praticamente completo. Modelou a maior parte dos dados, mas com pequenas omissões de observações secundárias.",
            "3": "Regular: Faltaram recursos críticos (como MedicationRequest ou Condition importantes), mas a estrutura básica do paciente está correta.",
            "2": "Ruim: O Bundle tem graves falhas de codificação médica ou ignorou a maioria dos dados do exame clínico.",
            "1": "Inaceitável: Resposta incompleta, JSON corrompido ou dados completamente incompatíveis com o paciente."
        }
    )
)

# Métrica de Função Python Customizada
syntax_quality_metric = types.Metric(
    name="fhir_syntax_validation",
    custom_function=validate_fhir_bundle_syntax
)

all_metrics = [
    fhir_clinical_rubric,
    syntax_quality_metric,
]

print(f"Configuradas {len(all_metrics)} métricas para avaliação.")

Configuradas 2 métricas para avaliação.


In [42]:
print("Iniciando avaliação no Gen AI Service...")
print("Nota: Este processo pode levar entre 1 e 3 minutos, pois o juiz inteligente avalia cada caso detalhadamente.")

eval_result = client.evals.evaluate(
    dataset=df,
    metrics=all_metrics
)

print("Avaliação concluída!")

Iniciando avaliação no Gen AI Service...
Nota: Este processo pode levar entre 1 e 3 minutos, pois o juiz inteligente avalia cada caso detalhadamente.


Computing Metrics for Evaluation Dataset: 100%|██████████| 6/6 [00:20<00:00,  3.38s/it]

Avaliação concluída!


In [43]:
eval_result.show()

In [44]:
def format_eval_case_to_markdown(case_result) -> str:
    """
    Formata um objeto de resultado de caso individual (EvalCaseResult) 
    em Markdown rico para exibição em tela.
    """
    # Índice do caso (ajustado para exibição humana a partir de 1)
    case_num = case_result.eval_case_index + 1
    output = [f"### Caso de Avaliação #{case_num}\n"]
    
    for cand_result in case_result.response_candidate_results:
        # Se houver apenas 1 candidato, essa identificação pode ser implícita
        cand_num = cand_result.response_index + 1
        output.append(f"#### Resposta Candidata {cand_num}")
        
        for metric_name, metric_res in cand_result.metric_results.items():
            # Nome amigável para as métricas
            friendly_metric_name = metric_name.replace('_', ' ').title()
            output.append(f"##### {friendly_metric_name}")
            
            if metric_res.error_message:
                output.append(f"❌ **Erro na Avaliação:** {metric_res.error_message}\n")
            else:
                # Formata a pontuação (ex: 2.0 ou 0.0)
                score_display = f"⭐ **Nota:** {int(metric_res.score)}/5" if metric_res.score is not None else "N/A"
                output.append(score_display)
                
                if metric_res.explanation:
                    # Indenta a explicação para que a hierarquia no markdown se mantenha
                    formatted_explanation = "\n".join(
                        f"> {line}" for line in metric_res.explanation.strip().split('\n')
                    )
                    output.append(f"\n**Análise do Avaliador:**\n{formatted_explanation}\n")
            
            output.append("---") # Linha divisória entre as métricas
            
    return "\n".join(output)

In [45]:
print(format_eval_case_to_markdown(eval_result.eval_case_results[0]))

### Caso de Avaliação #1

#### Resposta Candidata 1
##### Fhir Clinical Completeness
⭐ **Nota:** 2/5

**Análise do Avaliador:**
> A resposta gerada apresenta um Bundle FHIR válido e bem-formado, utilizando os recursos corretos (Patient, Encounter, Condition, MedicationRequest) e os sistemas de codificação padrão (SNOMED-CT para condição, RxNorm para medicação). Contudo, a fidelidade clínica dos dados é significativamente comprometida devido a várias omissões críticas. 
> 
> Em termos de 'Fidelidade de Dados', o Bundle falha em converter:
> - Todas as observações de sintomas mencionadas na queixa principal (ardência ao urinar, aumento da frequência urinária, negação de febre ou dor lombar).
> - Todas as observações do exame físico (afebril, abdômen indolor à palpação).
> - Detalhes cruciais da prescrição médica, como dose (100mg), via (VO), frequência (a cada 6h) e duração (por 5 dias) para a Nitrofurantoína. O MedicationRequest inclui apenas o nome do medicamento.
> Essas omissões repr